# 315 — Anomaly Detection

Runs all 6 rule-based Cypher anomaly patterns and displays real findings.

| Pattern | Expected findings |
|---|---|
| `transaction_structuring` | ACC-0596 receiving 6 sub-$10k suspicious transfers |
| `high_lvr_loans` | LOAN-0002 (LVR=95), LOAN-0013 (LVR=92) |
| `high_risk_industry` | BRW-0624, BRW-0627 (Gambling, IND-9530) |
| `layered_ownership` | BRW-0582→BRW-0581→BRW-0580→BRW-0579 chain |
| `high_risk_jurisdiction` | Borrowers in JUR-VU / JUR-MM / JUR-KH |
| `guarantor_concentration` | Borrowers guaranteeing 2+ loans |

In [ ]:
%run 311_agent_setup.ipynb

In [ ]:
from src.agent.anomaly_detector import AnomalyDetector
from IPython.display import HTML, display
import json

detector = AnomalyDetector(conn)

SEVERITY_COLOUR = {'HIGH': '#f8d7da', 'MEDIUM': '#fff3cd', 'LOW': '#d1ecf1'}

## Run all patterns

In [ ]:
all_findings = detector.run_all()
print(f'\nTotal anomaly patterns with findings: {len(all_findings)}')
for f in all_findings:
    print(f'  [{f.severity}] {f.pattern_name}: {len(f.evidence)} rows | entities: {f.entity_ids[:3]}')

## Pattern 1: Transaction Structuring

In [ ]:
f = detector.run('transaction_structuring')
print(f'Severity: {f.severity}')
print(f'Description: {f.description}')
print(f'\nFindings ({len(f.evidence)}):')
for row in f.evidence:
    print(f'  Target: {row.get("target_account")} | '
          f'Tx count: {row.get("tx_count")} | '
          f'Total AUD: {row.get("total_amount_aud")} | '
          f'Sources: {row.get("source_accounts")}')

## Pattern 2: High LVR Loans

In [ ]:
f = detector.run('high_lvr_loans')
print(f'Severity: {f.severity} — APG-223-THR-008 (LVR >= 90)')
print(f'\nBreaches ({len(f.evidence)}):')
for row in f.evidence:
    print(f'  {row.get("loan_id")} | LVR={row.get("lvr")}% | '
          f'Amount=AUD {row.get("amount_aud")} | '
          f'Borrower: {row.get("borrower_name")} ({row.get("borrower_risk")} risk)')

## Pattern 3: High-Risk Industry

In [ ]:
f = detector.run('high_risk_industry')
print(f'Severity: {f.severity}')
print(f'\nFindings ({len(f.evidence)}):')
for row in f.evidence:
    print(f'  {row.get("borrower_id")} | {row.get("name")} | '
          f'Industry: {row.get("industry_name")} | '
          f'AML sens: {row.get("aml_sensitivity")} | '
          f'Loans: {row.get("loan_ids")}')

## Pattern 4: Layered Ownership

In [ ]:
f = detector.run('layered_ownership')
print(f'Severity: {f.severity}')
print(f'\nDeepest ownership chains ({len(f.evidence)} total):')
for row in f.evidence[:5]:
    chain = ' → '.join(row.get('ownership_chain', []))
    pcts  = row.get('pct_chain', [])
    print(f'  Depth={row.get("chain_depth")} | {chain} | {pcts}%')

## Pattern 5: High-Risk Jurisdiction

In [ ]:
f = detector.run('high_risk_jurisdiction')
print(f'Severity: {f.severity}')
print(f'\nFindings ({len(f.evidence)}):')
for row in f.evidence[:10]:
    print(f'  {row.get("borrower_id")} | {row.get("name")} | '
          f'{row.get("link_type")} → {row.get("jurisdiction_id")} '
          f'({row.get("jurisdiction_name")}, {row.get("aml_risk_rating")})')

## Pattern 6: Guarantor Concentration

In [ ]:
f = detector.run('guarantor_concentration')
print(f'Severity: {f.severity}')
print(f'\nGuarantor concentration findings ({len(f.evidence)}):')
for row in f.evidence[:10]:
    print(f'  {row.get("borrower_id")} | {row.get("name")} | '
          f'Guaranteeing {row.get("guarantor_degree")} loans | '
          f'Total AUD {row.get("total_guaranteed_aud")}')

## Summary dashboard

In [ ]:
rows_html = ''
for f in all_findings:
    colour = SEVERITY_COLOUR.get(f.severity, '#fff')
    rows_html += f"""
    <tr style="background:{colour}">
      <td><b>{f.pattern_name}</b></td>
      <td><b>{f.severity}</b></td>
      <td>{len(f.evidence)}</td>
      <td>{', '.join(f.entity_ids[:4])}</td>
    </tr>"""

html = f"""
<table border='1' cellpadding='6' style='border-collapse:collapse;font-size:13px'>
  <tr><th>Pattern</th><th>Severity</th><th>Findings</th><th>Sample Entity IDs</th></tr>
  {rows_html}
</table>"""
display(HTML(html))